In [67]:
import os
import torch
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

In [68]:
print(os.listdir("/kaggle/input/competitions"))

['histopathologic-cancer-detection']


In [69]:
for item in os.listdir("/kaggle/input/competitions"):
    print(item)

histopathologic-cancer-detection


In [70]:
path = "/kaggle/input/competitions/histopathologic-cancer-detection"

for file in os.listdir(path):
    print(file)

sample_submission.csv
train_labels.csv
test
train


In [71]:
train_path = "/kaggle/input/competitions/histopathologic-cancer-detection/train"

files = os.listdir(train_path)

print("Number of images:", len(files))
print("First 10 files:", files[:10])

Number of images: 220025
First 10 files: ['d43c081bafa286f9c1f7e921883f26ceafebc912.tif', '092d0eedebce504847715ee046b6ad74b57599b4.tif', 'b0d2582c6218a8764323fc940b41312282b99bf4.tif', '187c99df762f13f99818e5593d4bab4c6577e7e3.tif', '7c5270c83837de5a5cbb2dca511559dc39d19d53.tif', '5a32933e093185f5fc91d30fc83ad571c6818d25.tif', '42e77d193e73811e0bb65a0cbd9b01c5c27900fa.tif', '27bb898f54a0b9345f6c4a9083299e4465860861.tif', '89cd55e4300440612347c38f306da688a166fd40.tif', 'cd600f77aa2af7c93dc6cd836e44edada3d8c403.tif']


In [72]:
labels_path = "/kaggle/input/competitions/histopathologic-cancer-detection/train_labels.csv"

df = pd.read_csv(labels_path)

print(df.head())
print()
print(df.columns)
print()
print(df["label"].value_counts())

                                         id  label
0  f38a6374c348f90b587e046aac6079959adf3835      0
1  c18f2d887b7ae4f6742ee445113fa1aef383ed77      1
2  755db6279dae599ebb4d39a9123cce439965282d      0
3  bc3f0c64fb968ff4a8bd33af6971ecae77c75e08      0
4  068aba587a4950175d04c680d38943fd488d6a9d      0

Index(['id', 'label'], dtype='object')

label
0    130908
1     89117
Name: count, dtype: int64


In [73]:
# مسیرها
labels_path = "/kaggle/input/competitions/histopathologic-cancer-detection/train_labels.csv"
train_path = "/kaggle/input/competitions/histopathologic-cancer-detection/train"

# خواندن labels
df = pd.read_csv(labels_path)

# انتخاب تصادفی 5000 نمونه از هر کلاس
class_0 = df[df["label"] == 0].sample(n=5000, random_state=42)
class_1 = df[df["label"] == 1].sample(n=5000, random_state=42)

# ترکیب دو کلاس
subset = pd.concat([class_0, class_1])

# به‌هم‌زدن ترتیب
subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)

print("Total:", len(subset))
print("\nClass distribution:")
print(subset["label"].value_counts())

Total: 10000

Class distribution:
label
1    5000
0    5000
Name: count, dtype: int64


In [74]:
class PCamDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["id"] + ".tif"
        )

        image = Image.open(image_path).convert("RGB")
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [75]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [76]:
dataset = PCamDataset(
    subset,
    "/kaggle/input/competitions/histopathologic-cancer-detection/train",
    transform=transform
)
image, label = dataset[0]

print("Image shape:", image.shape)
print("Label:", label)

Image shape: torch.Size([3, 96, 96])
Label: tensor(1.)


In [77]:
train_df, val_df = train_test_split(
    subset,
    test_size=0.2,
    random_state=42,
    stratify=subset["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\nTrain classes:")
print(train_df["label"].value_counts())

print("\nValidation classes:")
print(val_df["label"].value_counts())

Train: 8000
Validation: 2000

Train classes:
label
0    4000
1    4000
Name: count, dtype: int64

Validation classes:
label
0    1000
1    1000
Name: count, dtype: int64


In [78]:
train_dataset = PCamDataset(
    train_df,
    "/kaggle/input/competitions/histopathologic-cancer-detection/train",
    transform=transform
)

val_dataset = PCamDataset(
    val_df,
    "/kaggle/input/competitions/histopathologic-cancer-detection/train",
    transform=transform
)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2)

val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2)

In [79]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 96, 96])
Labels shape: torch.Size([32])


In [80]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 12 * 12, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [82]:
model = CNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
print(model)

CNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9216, out_features=64, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [84]:
# تعداد epoch
num_epochs = 5

for epoch in range(num_epochs):

    # ==========================================
    # TRAIN
    # ==========================================

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images).squeeze(1)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Predictions
        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        # Calculate accuracy
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        train_loss += loss.item()

    # Average training loss
    train_loss = train_loss / len(train_loader)

    # Training accuracy
    train_accuracy = train_correct / train_total


    # ==========================================
    # VALIDATION
    # ==========================================

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images).squeeze(1)

            # Calculate validation loss
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            # Predictions
            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            # Calculate accuracy
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    # Average validation loss
    val_loss = val_loss / len(val_loader)

    # Validation accuracy
    val_accuracy = val_correct / val_total


    # ==========================================
    # PRINT RESULTS
    # ==========================================

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )


# ==================================================
# FINAL MODEL EVALUATION
# ==================================================

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        # Model output
        outputs = model(images).squeeze(1)

        # Convert logits to predictions
        predictions = (torch.sigmoid(outputs) >= 0.5).int()

        # Save predictions and labels
        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )


# ==================================================
# ACCURACY
# ==================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("\nFinal Accuracy:")
print(f"{accuracy:.4f}")


# ==================================================
# CONFUSION MATRIX
# ==================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)


# ==================================================
# CLASSIFICATION REPORT
# ==================================================

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

Epoch [1/5] Train Loss: 0.5311 Train Acc: 0.7555 Val Loss: 0.4756 Val Acc: 0.7805
Epoch [2/5] Train Loss: 0.5065 Train Acc: 0.7750 Val Loss: 0.4503 Val Acc: 0.8040
Epoch [3/5] Train Loss: 0.4852 Train Acc: 0.7839 Val Loss: 0.4256 Val Acc: 0.8105
Epoch [4/5] Train Loss: 0.4561 Train Acc: 0.7965 Val Loss: 0.4996 Val Acc: 0.7770
Epoch [5/5] Train Loss: 0.4428 Train Acc: 0.8125 Val Loss: 0.4010 Val Acc: 0.8250

Final Accuracy:
0.8250

Confusion Matrix:
[[880 120]
 [230 770]]

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.79      0.88      0.83      1000
     Class 1       0.87      0.77      0.81      1000

    accuracy                           0.82      2000
   macro avg       0.83      0.82      0.82      2000
weighted avg       0.83      0.82      0.82      2000

